> **Edición pública anonimizada.** Nombres y rutas comerciales se sustituyeron por etiquetas genéricas; no se incluyen bases ni modelos originales. Los resultados históricos se consultan en la tesis adaptada. La demostración pública usa datos y modelos sintéticos.


In [ ]:
from pathlib import Path
import os, sys

# Find the repository independently of the computer or clone location.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / ".tesis-portafolio").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Abre Jupyter desde la carpeta de este repositorio.")
os.chdir(REPO_ROOT / '04 Notebook Final')
sys.path.insert(0, str(REPO_ROOT / "01 Limpieza Código"))


# Notebook Final — Segmentación y Modelo Predictivo de Leads Inmobiliarios

## Entregable consolidado de tesis

Este notebook reúne, en un único flujo reproducible, los **tres componentes finales** del proyecto:

1. **Parte I — Limpieza y Preparación de Datos**: pipeline reproducible que construye el dataset final de leads a partir de las fuentes crudas.
2. **Parte II — Segmentación de Leads (K-Means)**: modelo de clustering final (configuración **R1**, k=6) que agrupa a los leads en seis perfiles demográfico-conductuales interpretables.
3. **Parte III — Modelo Predictivo de Conversión (Regresión Logística)**: modelo final que estima la probabilidad de conversión de cada lead (Universo H, 22,656 leads).

Cada parte proviene de un notebook de trabajo más amplio, en el que se exploraron y descartaron numerosas configuraciones alternativas (distintos conjuntos de variables, algoritmos, ponderaciones y estrategias de balanceo). **Aquí se muestran únicamente las configuraciones finales seleccionadas**, con el código necesario para reproducirlas y modificarlas, y con explicaciones dirigidas a un público no necesariamente técnico. Donde corresponde, se resume brevemente el proceso de búsqueda que llevó a la configuración final, sin reproducir el detalle de cada experimento descartado.


---
# Parte I — Limpieza y Preparación de Datos

Esta parte reproduce, prácticamente sin cambios, el notebook `01_limpieza_pipeline_thesis.ipynb` de la carpeta `01 Limpieza Código/`. Ese notebook ya tiene el formato de un apéndice de tesis: ejecuta los scripts canónicos del pipeline de limpieza (la fuente de verdad) y documenta sus resultados, sin experimentos fallidos ni callejones sin salida.

La única diferencia respecto al original es la celda inmediatamente siguiente, que ajusta el directorio de trabajo para que las rutas relativas del pipeline (`scripts/`, `data/`, `reports/`) apunten correctamente a `01 Limpieza Código/`, ya que este notebook consolidado vive en una carpeta distinta.


In [ ]:
import os

# Este notebook consolidado vive en "04 Notebook Final/". El pipeline de
# limpieza (scripts, datos y reportes) usa rutas relativas a su propia
# carpeta "01 Limpieza Código/", así que nos movemos temporalmente allí
# para que el resto de las celdas de la Parte I (copiadas sin cambios del
# notebook original) funcionen sin modificaciones.
_cwd_original = os.getcwd()
os.chdir(f'{REPO_ROOT}/01 Limpieza Código')
print(f'Directorio de trabajo para la Parte I: {os.getcwd()}')


# Pipeline reproducible de preparación de datos
## Apéndice de tesis — Limpieza de datos de leads e ingeniería de variables

Este notebook documenta y coordina el pipeline completo de preparación de datos utilizado en la tesis. Su enfoque es **priorizar la documentación**: los scripts de Python ubicados en `scripts/` permanecen como la fuente de verdad, mientras que este notebook los ejecuta en secuencia y resume los artefactos que generan.

> **Regla de reproducibilidad:** ejecute este notebook desde la raíz de `01 Limpieza Código` mediante **Kernel → Restart Kernel and Run All Cells**. Todas las rutas son relativas a la raíz del proyecto.

### Orden del pipeline

1. Comprobaciones del entorno
2. Armonización de nuevas exportaciones
3. Construcción del dataset raw final
4. Limpieza del dataset final
5. Construcción de variables finales
6. Validación de los outputs finales
7. Ejecución del análisis exploratorio de datos (EDA) final

Los outputs destinados al modelado excluyen deliberadamente las variables posteriores al resultado que podrían provocar fuga de información (*target leakage*). Se conservan archivos paralelos con dichas variables únicamente para auditoría y trazabilidad.


## Cómo leer este apéndice

Cada etapa contiene cuatro apartados explícitos: **Propósito**, **Inputs**, **Procesamiento realizado** y **Outputs generados**. A continuación, la celda de código ejecuta el script correspondiente como un proceso de Python independiente. Esto conserva el mismo contexto de ejecución que el pipeline de línea de comandos y evita que el estado del notebook modifique silenciosamente la lógica de producción.

Las celdas de diagnóstico no reconstruyen ningún output. Cargan los archivos escritos por los scripts y muestran dimensiones, valores faltantes, duplicados, balance de la variable objetivo, distribuciones categóricas y ejemplos seleccionados. Si un script fuente falla, la ejecución se detiene inmediatamente para impedir que resúmenes antiguos se interpreten como resultados actuales.


## Configuración del notebook

Las utilidades siguientes se limitan a la coordinación y presentación de resultados. No reproducen reglas de limpieza, definiciones de variables, uniones ni lógica de validación de los scripts fuente. Únicamente se emplean la biblioteca estándar de Python, pandas, openpyxl y las utilidades de visualización de Jupyter.


In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display
from scripts.schema_contract import (
    FORBIDDEN_ANALYTICAL_COLUMNS,
    REQUIRED_ANALYTICAL_LABEL_COLUMNS,
    validate_analytical_label_schema,
)

PROJECT_ROOT = Path.cwd()
SCRIPTS_DIR = Path("scripts")
VENV_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"
PIPELINE_PYTHON = VENV_PYTHON if VENV_PYTHON.exists() else Path(sys.executable)

required_scripts = [
    "check_environment.py",
    "01_harmonize_new_exports.py",
    "01b_harmonize_pt3_exports.py",
    "02_build_final_raw_dataset.py",
    "03_clean_final_dataset.py",
    "04_build_final_features.py",
    "05_validate_final_outputs.py",
    "06_run_final_eda.py",
]
missing_scripts = [name for name in required_scripts if not (SCRIPTS_DIR / name).is_file()]
if missing_scripts:
    raise RuntimeError(
        "El notebook debe ejecutarse desde la raíz de '01 Limpieza Código'. "
        f"Scripts faltantes: {missing_scripts}. Directorio actual: {PROJECT_ROOT}"
    )

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Ejecutable del notebook: {sys.executable}")
print(f"Ejecutable del pipeline: {PIPELINE_PYTHON}")
print(f"Ejecución iniciada: {datetime.now():%Y-%m-%d %H:%M:%S}")

In [ ]:
def run_script(script_name):
    """Ejecuta un script canónico del pipeline desde la raíz del proyecto."""
    script_path = SCRIPTS_DIR / script_name
    display(Markdown(f"**Ejecutando la fuente de verdad:** `{script_path.as_posix()}`"))
    completed = subprocess.run(
        [str(PIPELINE_PYTHON), str(script_path)],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print(f"Finalizado con código de salida {completed.returncode}.")


def load_csv(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"No se encontró el artefacto esperado del pipeline: {path}")
    return pd.read_csv(path, low_memory=False)


def dataset_summary(df, name, id_candidates=("id_trato", "id_Trato")):
    id_col = next((col for col in id_candidates if col in df.columns), None)
    duplicate_rows = int(df.duplicated().sum())
    duplicate_ids = int(df[id_col].duplicated().sum()) if id_col else pd.NA
    overview = pd.DataFrame(
        {
            "métrica": ["filas", "variables", "celdas faltantes", "celdas faltantes (%)", "filas duplicadas", f"valores duplicados de {id_col or 'ID'}"],
            "valor": [
                len(df),
                df.shape[1],
                int(df.isna().sum().sum()),
                round(df.isna().sum().sum() / max(df.size, 1) * 100, 2),
                duplicate_rows,
                duplicate_ids,
            ],
        }
    )
    display(Markdown(f"### {name}"))
    display(overview)

    missing = df.isna().sum().sort_values(ascending=False)
    missing = missing[missing.gt(0)].head(15).rename("faltantes").to_frame()
    if not missing.empty:
        missing["porcentaje"] = (missing["faltantes"] / len(df) * 100).round(2)
        display(Markdown("**Variables con más valores faltantes (15 principales)**"))
        display(missing)
    else:
        display(Markdown("**Valores faltantes:** no se detectaron valores faltantes."))


def show_distribution(df, column, top=20):
    if column not in df.columns:
        print(f"Columna no disponible: {column}")
        return
    counts = df[column].value_counts(dropna=False).head(top).rename("cantidad").to_frame()
    counts["porcentaje"] = (counts["cantidad"] / len(df) * 100).round(2)
    display(Markdown(f"**Distribución: `{column}`**"))
    display(counts)


def artifact_inventory(paths):
    rows = []
    for item in map(Path, paths):
        rows.append(
            {
                "artefacto": item.as_posix(),
                "existe": item.is_file(),
                "tamaño_kb": round(item.stat().st_size / 1024, 1) if item.is_file() else pd.NA,
                "modificado": datetime.fromtimestamp(item.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S") if item.is_file() else pd.NA,
            }
        )
    result = pd.DataFrame(rows)
    display(result)
    return result


def show_markdown_report(path, max_characters=12000):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    text = path.read_text(encoding="utf-8")
    suffix = "\n\n*Vista previa truncada en el notebook.*" if len(text) > max_characters else ""
    display(Markdown(text[:max_characters] + suffix))


def show_excel_sheet(path, sheet_name, rows=10):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    display(pd.read_excel(path, sheet_name=sheet_name).head(rows))

# 1. Comprobaciones del entorno

### Propósito
Confirmar que los paquetes de Python requeridos y la estructura de directorios estén disponibles antes de modificar cualquier dato. Fallar anticipadamente es importante en un pipeline reproducible porque evita generar outputs parcialmente actualizados.

### Inputs
El entorno activo de Python, `requirements.txt`, el directorio `scripts/` y los directorios esperados de datos raw.

### Procesamiento realizado
El script canónico de comprobación del entorno verifica pandas, NumPy, openpyxl y las carpetas principales del proyecto. Después, el notebook inventaría los archivos raw sin analizarlos ni transformarlos.

### Outputs generados
No se genera ningún dataset. El resultado visible en consola y el inventario de archivos constituyen el registro previo a la ejecución.


In [ ]:
run_script("check_environment.py")

raw_inputs = sorted(Path("data/raw").glob("Leads UVG *.xlsx"))
raw_inputs += sorted(Path("data/raw/data_pt2").glob("Leads_*_UVG.csv"))
raw_inputs += [
    Path("data/raw/Base Historica por Producto - fuente_catalogo_a.xlsx"),
    Path("data/raw/Proyectos fuente_catalogo_b-2024.xlsx"),
]
preflight = artifact_inventory(raw_inputs)
if not preflight["existe"].all():
    missing = preflight.loc[~preflight["existe"], "artefacto"].tolist()
    raise FileNotFoundError(f"Faltan inputs raw obligatorios: {missing}")

# 2. Armonización de nuevas exportaciones

### Propósito
Estandarizar las exportaciones de proyectos más recientes antes de incorporarlas a las fuentes históricas en Excel. La armonización resuelve diferencias de esquema y etiquetas en el límite de ingestión, lo que reduce el riesgo de tratar variables equivalentes como columnas distintas.

### Inputs
Archivos delimitados por punto y coma que coinciden con `data/raw/data_pt2/Leads_*_UVG.csv`.

### Procesamiento realizado
`01_harmonize_new_exports.py` identifica las exportaciones y homologa cada par bajo el nombre `*_label`: prioriza el valor label cuando existe y completa sus nulos con la columna original; si un archivo solo contiene la columna original, la renombra al nombre label canónico antes de la unión. También elimina `tipomodelo_label`, normaliza nombres clave del esquema y etiquetas de proyecto, conserva el identificador de la fuente original y valida la conservación de filas y la presencia de etapas requeridas.

### Outputs generados
Archivos CSV armonizados por proyecto, `pt2_all_harmonized.csv` y un reporte en Excel que documenta los reemplazos de etiquetas y su cobertura.


In [ ]:
run_script("01_harmonize_new_exports.py")

In [ ]:
pt2_path = Path("data/raw/data_pt2/harmonized/pt2_all_harmonized.csv")
pt2 = load_csv(pt2_path)
dataset_summary(pt2, "Nuevas exportaciones armonizadas")
show_distribution(pt2, "Proyecto")
show_distribution(pt2, "Etapa Trato")

harmonization_outputs = sorted(Path("data/raw/data_pt2/harmonized").glob("*.csv"))
harmonization_outputs.append(Path("reports/final/label_replacement_report.xlsx"))
artifact_inventory(harmonization_outputs)
show_excel_sheet("reports/final/label_replacement_report.xlsx", "label_replacement", rows=12)

# 2b. Armonización y fusión de la reexportación PT3

### Propósito
Incorporar `data/raw/data_pt3/`: reexportaciones más recientes del CRM para proyecto03, proyecto04 y proyecto05 (mismo esquema de columnas "bonitas" que los Excel base proyecto01/proyecto02, sin el formato código+label de PT2) más un proyecto nuevo, proyecto06. PT2 tiene un problema de calidad conocido: `Fecha_Creación_Trato` queda constante (un mismo valor) en todas sus filas para proyecto03, proyecto04 y proyecto05, mientras que PT3 trae fechas reales.

### Regla de fusión con PT2
Para cada `id_Trato` presente en PT3 se conserva la fila de PT3 (fuente más reciente/confiable). Los `id_Trato` que sólo existen en PT2 se conservan sin cambios. Cuando los conjuntos de `id_Trato` de PT2 y PT3 coinciden por completo para un proyecto, el resultado equivale a usar sólo PT3; cuando difieren, el resultado es una unión (join). El detalle por proyecto se guarda en `reports/final/pt3_merge_summary.md`.

### Inputs
Archivos `data/raw/data_pt3/Reporte UVG *.xlsx` y el PT2 armonizado producido en la etapa 2.

### Outputs generados
CSV armonizados por proyecto y `pt3_all_harmonized.csv` en `data/raw/data_pt3/harmonized/`, el archivo fusionado `data/raw/data_pt2/harmonized/pt2_pt3_merged.csv` (input de la etapa 3) y `reports/final/pt3_merge_summary.md`.

In [ ]:
run_script("01b_harmonize_pt3_exports.py")

In [ ]:
pt3_merged_path = Path("data/raw/data_pt2/harmonized/pt2_pt3_merged.csv")
pt3_merged = load_csv(pt3_merged_path)
dataset_summary(pt3_merged, "PT2 + PT3 fusionados")
show_distribution(pt3_merged, "Proyecto")
show_distribution(pt3_merged, "Etapa Trato")

pt3_outputs = sorted(Path("data/raw/data_pt3/harmonized").glob("*.csv"))
pt3_outputs.append(pt3_merged_path)
artifact_inventory(pt3_outputs)
show_markdown_report("reports/final/pt3_merge_summary.md")

# 3. Construcción del dataset raw final

### Propósito
Crear una única base raw trazable que contenga tanto el historial consolidado de proyectos como las nuevas exportaciones armonizadas. En esta etapa el énfasis es la integración de datos, no el modelado predictivo.

### Inputs
Archivos históricos que coinciden con `data/raw/Leads UVG *.xlsx` y el archivo PT2+PT3 fusionado producido en la etapa 2b.

### Procesamiento realizado
`02_build_final_raw_dataset.py` alinea los esquemas, agrega la procedencia de cada fuente, concatena los registros y resuelve identificadores de negocio repetidos conservando la fila más completa. También comprueba que los archivos fuente base no hayan sido modificados y que todos los proyectos requeridos permanezcan representados.

### Outputs generados
`data/raw/leads_final_combined.csv`, el input raw para la limpieza posterior.


In [ ]:
run_script("02_build_final_raw_dataset.py")

In [ ]:
raw_final_path = Path("data/raw/leads_final_combined.csv")
raw_final = load_csv(raw_final_path)
dataset_summary(raw_final, "Dataset raw final")
show_distribution(raw_final, "Proyecto")
show_distribution(raw_final, "Etapa Trato")
artifact_inventory([raw_final_path])

# 4. Limpieza del dataset final

### Propósito
Convertir la tabla raw integrada en un dataset consistente y apto para modelado. La limpieza mejora la comparabilidad entre fuentes, controla observaciones duplicadas, define la variable objetivo del aprendizaje supervisado y elimina información que no estaría disponible en el momento de realizar una predicción.

### Inputs
`data/raw/leads_final_combined.csv` de la etapa 3 y las reglas reutilizables de `scripts/cleaning_helpers.py`.

### Procesamiento realizado
`03_clean_final_dataset.py` diagnostica la calidad de los datos; construye y valida `target_won`; elimina variables con fuga de información, con ausencia extrema o estructuralmente inadecuadas; normaliza texto categórico, campañas, fechas, edad, proyectos, datos de contacto y nombres de columnas; elimina registros de prueba; y deduplica negocios mediante la lógica de prioridad documentada.

### Outputs generados
Un dataset limpio apto para modelado, un dataset paralelo de auditoría con variables de fuga, una tabla que contiene únicamente variables predictoras, una referencia de fuga y reportes detallados de calidad, normalización, diccionario de variables, columnas eliminadas y resumen en Markdown.


In [ ]:
run_script("03_clean_final_dataset.py")

In [ ]:
clean_path = Path("data/processed/leads_cleaned_final.csv")
clean_with_leakage_path = Path("data/processed/leads_cleaned_with_leakage_final.csv")
clean = load_csv(clean_path)
clean_with_leakage = load_csv(clean_with_leakage_path)
validate_analytical_label_schema(clean, clean_path.name)
validate_analytical_label_schema(clean_with_leakage, clean_with_leakage_path.name)

dataset_summary(clean, "Dataset limpio apto para modelado")
show_distribution(clean, "target_won")
show_distribution(clean, "proyecto")
for variable in ["tiene_hijos", "sexo", "estado_civil"]:
    show_distribution(clean, variable, top=12)

comparison = pd.DataFrame(
    {
        "dataset": ["raw final", "limpio y apto para modelado", "auditoría limpia con fuga"],
        "filas": [len(raw_final), len(clean), len(clean_with_leakage)],
        "variables": [raw_final.shape[1], clean.shape[1], clean_with_leakage.shape[1]],
    }
)
display(Markdown("**Cambios de dimensiones durante la limpieza**"))
display(comparison)

example_columns = [col for col in ["id_trato", "proyecto", "campania", "edad", "tiene_hijos", "target_won"] if col in clean.columns]
display(Markdown("**Ejemplos de variables limpias**"))
display(clean[example_columns].head(10))

In [ ]:
cleaning_artifacts = [
    clean_path,
    clean_with_leakage_path,
    Path("data/processed/leads_features_final.csv"),
    Path("data/processed/leads_leakage_reference_final.csv"),
    Path("reports/final/data_quality_report.xlsx"),
    Path("reports/final/normalization_before_after.xlsx"),
    Path("reports/final/variable_dictionary.xlsx"),
    Path("reports/final/columns_removed.csv"),
    Path("reports/final/cleaning_summary.md"),
]
artifact_inventory(cleaning_artifacts)
show_markdown_report("reports/final/cleaning_summary.md")

# 5. Construcción de variables finales

### Propósito
Transformar los campos limpios en predictores interpretables y enriquecer los leads con contexto a nivel de proyecto. La ingeniería de variables proporciona representaciones estables para los algoritmos de aprendizaje automático y, al mismo tiempo, mantiene definiciones explícitas y auditables.

### Inputs
Los datasets limpios de la etapa 4, `scripts/feature_helpers.py`, la tabla de referencia de proyectos y los dos libros externos del mercado inmobiliario ubicados en `data/raw/`.

### Procesamiento realizado
`04_build_final_features.py` deriva la plataforma, el tipo y la audiencia de campaña; crea rangos de edad orientados al modelado; audita las fuentes externas; construye o sincroniza atributos de proyecto; enriquece tanto el dataset modelable como el dataset de auditoría con fuga; y ejecuta comprobaciones específicas de las variables creadas.

### Outputs generados
La tabla enriquecida final para modelado, su equivalente de auditoría, la referencia sincronizada de proyectos y los diagnósticos, mapeos, diccionarios y reportes Markdown de ingeniería de variables.


In [ ]:
run_script("04_build_final_features.py")

In [ ]:
enriched_path = Path("data/features/leads_enriched_final.csv")
enriched_audit_path = Path("data/features/leads_enriched_with_leakage_final.csv")
enriched = load_csv(enriched_path)
enriched_audit = load_csv(enriched_audit_path)
validate_analytical_label_schema(enriched, enriched_path.name)
validate_analytical_label_schema(enriched_audit, enriched_audit_path.name)

dataset_summary(enriched, "Dataset enriquecido final para modelado")
show_distribution(enriched, "target_won")
for variable in ["campania_plataforma", "campania_tipo", "campania_audiencia", "rango_edad_modelo", "zona_proyecto"]:
    show_distribution(enriched, variable, top=15)

new_features = [col for col in enriched.columns if col not in clean.columns]
display(Markdown(f"**Nuevas variables agregadas en la etapa 5 ({len(new_features)}):** `{'`, `'.join(new_features)}`"))
example_features = [col for col in ["proyecto", "campania", "campania_plataforma", "campania_tipo", "campania_audiencia", "edad", "rango_edad_modelo", "zona_proyecto", "precio_m2_promedio"] if col in enriched.columns]
display(Markdown("**Ejemplos de variables creadas y enriquecidas**"))
display(enriched[example_features].head(12))

In [ ]:
feature_artifacts = [
    enriched_path,
    enriched_audit_path,
    Path("data/reference/proyectos_atributos.csv"),
    Path("reports/final/fe_diagnostic_report.xlsx"),
    Path("reports/final/campania_mapping.xlsx"),
    Path("reports/final/fe_variable_dictionary_update.xlsx"),
    Path("reports/final/external_files_mapping.md"),
    Path("reports/final/fe_summary.md"),
]
artifact_inventory(feature_artifacts)
show_markdown_report("reports/final/fe_summary.md")

# 6. Validación de los outputs finales

### Propósito
Aplicar criterios explícitos de aceptación antes de utilizar el dataset para EDA o modelado. La validación convierte supuestos —como integridad del objetivo, conservación de filas, ausencia de duplicados y separación de variables con fuga— en comprobaciones ejecutables.

### Inputs
Todos los artefactos raw, limpios, de auditoría, enriquecidos y de resumen producidos en las etapas 3 a 5.

### Procesamiento realizado
`05_validate_final_outputs.py` comprueba la existencia y contenido de los archivos, los valores binarios del objetivo, la conservación de filas durante el enriquecimiento, los proyectos y variables obligatorios, la unicidad de identificadores de negocio, la clasificación de las etapas designadas como ganadas y la separación de variables con fuga de los archivos modelables.

### Outputs generados
Un registro visible de validación aprobada o fallida y `reports/final/eda_ready_summary.md`. Cualquier aserción fallida detiene el pipeline antes del EDA.


In [ ]:
run_script("05_validate_final_outputs.py")

In [ ]:
validation_observations = pd.DataFrame(
    {
        "comprobación": [
            "target_won contiene únicamente 0/1",
            "target_won no contiene valores faltantes",
            "id_trato es único",
            "las filas del dataset limpio y enriquecido coinciden",
            "las filas de auditoría limpia y enriquecida coinciden",
            "no existen columnas de código ni tipomodelo_label",
            "se conservan todas las columnas label requeridas",
        ],
        "observado": [
            set(enriched["target_won"].dropna().unique()).issubset({0, 1}),
            enriched["target_won"].notna().all(),
            not enriched["id_trato"].duplicated().any(),
            len(clean) == len(enriched),
            len(clean_with_leakage) == len(enriched_audit),
            not (FORBIDDEN_ANALYTICAL_COLUMNS & set(enriched.columns)),
            REQUIRED_ANALYTICAL_LABEL_COLUMNS.issubset(enriched.columns),
        ],
    }
)
display(validation_observations)
artifact_inventory([Path("reports/final/eda_ready_summary.md")])
show_markdown_report("reports/final/eda_ready_summary.md")

# 7. Ejecución del análisis exploratorio de datos final

### Propósito
Caracterizar el dataset validado y apto para modelado antes de aplicar clustering o aprendizaje supervisado. El EDA permite identificar el desbalance de clases, los valores faltantes, la cardinalidad, los rangos numéricos, el soporte de las categorías y posibles riesgos de modelado.

### Inputs
`data/features/leads_enriched_final.csv`, que excluye fuga posterior al resultado y aprobó la etapa 6.

### Procesamiento realizado
`06_run_final_eda.py` construye tablas de descripción general del dataset, variable objetivo, valores faltantes, cardinalidad, variables numéricas, asociaciones con el objetivo, alertas de soporte y recomendaciones. El script verifica que su input no haya sido modificado y no crea subconjuntos de entrenamiento.

### Outputs generados
`reports/eda/eda_technical_report.xlsx` y `reports/eda/eda_summary.md`. El notebook muestra hojas seleccionadas de estos reportes en lugar de recalcular el análisis.


In [ ]:
run_script("06_run_final_eda.py")

In [ ]:
eda_xlsx = Path("reports/eda/eda_technical_report.xlsx")
eda_md = Path("reports/eda/eda_summary.md")
artifact_inventory([eda_xlsx, eda_md])

display(Markdown("### Descripción general del dataset para EDA"))
show_excel_sheet(eda_xlsx, "dataset_overview", rows=30)
display(Markdown("### Balance de la variable objetivo"))
show_excel_sheet(eda_xlsx, "target_balance", rows=10)
display(Markdown("### Variables con mayor cantidad de valores faltantes"))
show_excel_sheet(eda_xlsx, "missingness", rows=15)
display(Markdown("### Recomendaciones del EDA"))
show_excel_sheet(eda_xlsx, "eda_recommendations", rows=30)

In [ ]:
show_markdown_report(eda_md)

# Registro de reproducibilidad y entrega

Una ejecución exitosa significa que las siete secciones terminaron sin excepciones y que la validación final informó que todas las comprobaciones fueron aprobadas. El principal input para modelado es:

- `data/features/leads_enriched_final.csv` — enriquecido, apto para modelado y libre de fuga de información posterior al resultado.

Los archivos equivalentes que contienen variables con fuga son artefactos de auditoría y no deben utilizarse como matrices de predictores. Los reportes técnicos en Excel y Markdown conservan evidencia detallada para la revisión metodológica, mientras que los scripts originales permanecen como la implementación autoritativa de cada transformación.


In [ ]:
final_record = pd.DataFrame(
    {
        "elemento": ["finalizado_el", "python", "dataset_apto_para_modelado", "filas", "variables", "faltantes_objetivo", "ids_duplicados"],
        "valor": [
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            sys.version.split()[0],
            enriched_path.as_posix(),
            len(enriched),
            enriched.shape[1],
            int(enriched["target_won"].isna().sum()),
            int(enriched["id_trato"].duplicated().sum()),
        ],
    }
)
display(final_record)
display(Markdown("## Pipeline completado exitosamente"))

In [ ]:
# Volvemos al directorio de trabajo original antes de continuar con la
# Parte II, que usa rutas absolutas propias.
os.chdir(_cwd_original)
print(f'Directorio de trabajo restaurado: {os.getcwd()}')


---
# Parte II — Segmentación de Leads (K-Means)

Esta parte reproduce el **modelo final de clustering** obtenido en `kmeans_leads_analysis.ipynb` (carpeta `02 K Means/`), un notebook de trabajo de más de 400 celdas que exploró numerosas configuraciones antes de llegar al modelo seleccionado.

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué se hizo antes de llegar aquí (resumen):</strong> se probaron experimentos con distintos universos de leads, variables (adquisición, perfil demográfico, variables económicas), transformaciones (estándar, robusta, PCA), esquemas de codificación y ponderaciones — más de veinte configuraciones documentadas, además de una búsqueda dirigida específicamente a encontrar segmentaciones demográfico-conductuales con Silhouette ≥ 0.50. La configuración <b>R1</b> (edad continua con peso reducido + sexo, estado civil, razón de compra y tipo de unidad ponderados, k=6) fue la que mejor equilibró calidad geométrica (Silhouette ≈ 0.518), tamaños de cluster balanceados (12.2%–20.4%) e interpretabilidad de negocio, sin usar variables económicas. Esta sección reconstruye <b>únicamente</b> ese modelo final, de forma autocontenida.</div>


In [ ]:
# ── Librerías utilizadas en la Parte II ───────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    pairwise_distances,
)
from scipy.stats import f_oneway, chi2_contingency, tukey_hsd, norm

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11})
sns.set_palette('Set2')

RANDOM_STATE_KM = 42
OUT_DIR_KM = Path(f'{REPO_ROOT}/02 K Means/Output/modelo_final_notebook')
OUT_DIR_KM.mkdir(parents=True, exist_ok=True)


## Preparación del universo de clustering

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Carga el dataset enriquecido de leads calificados (el mismo output final de la Parte I) y aplica el criterio de completitud del modelo final: solo se conservan los leads con las cinco variables demográfico-conductuales obligatorias completas.</div>


In [ ]:
DATA_PATH_KM = Path(f'{REPO_ROOT}/01 Limpieza Código/'
                    'data/features/leads_enriched_qualified_final.csv')

df_km_raw = pd.read_csv(DATA_PATH_KM, encoding='utf-8-sig', low_memory=False)
print(f'Leads calificados totales: {len(df_km_raw):,}')

VARS_OBLIGATORIAS = ['edad', 'sexo', 'estado_civil', 'razon_de_compra', 'tipo_unidad']

df_universo = df_km_raw.loc[df_km_raw[VARS_OBLIGATORIAS].notna().all(axis=1)].copy()
for col in ['sexo', 'estado_civil', 'razon_de_compra', 'tipo_unidad']:
    df_universo[col] = df_universo[col].astype(str)
df_universo['edad'] = pd.to_numeric(df_universo['edad'], errors='coerce')
df_universo = df_universo.dropna(subset=['edad']).reset_index(drop=True)

print(f'Universo de clustering (5 variables obligatorias completas): {len(df_universo):,} leads '
      f'({len(df_universo) / len(df_km_raw):.1%} del total calificado)')


## Configuración R1 y construcción de la matriz de clustering

La configuración **R1** representa a cada lead como: edad estandarizada con un peso reducido (0.25, para que no domine frente a las variables categóricas) y cuatro bloques *one-hot* balanceados (divididos entre la raíz cuadrada de su número de categorías, para que bloques con más niveles no pesen artificialmente más) y ponderados según su importancia sustantiva para el negocio.

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Define la función que construye la matriz numérica R1 a partir del universo de clustering y la ejecuta.</div>


In [ ]:
PESO_EDAD = 0.25
PESOS_CATEGORICAS_R1 = {
    'sexo': 1.2,
    'estado_civil': 1.5,
    'razon_de_compra': 1.4,
    'tipo_unidad': 1.0,
}
CAT_VARS_R1 = ['sexo', 'estado_civil', 'razon_de_compra', 'tipo_unidad']
K_R1 = 6


def construir_matriz_r1(df):
    """Construye la matriz de features de la configuración R1: edad
    estandarizada y ponderada + bloques OHE balanceados y ponderados."""
    bloques = [StandardScaler().fit_transform(df[['edad']]) * PESO_EDAD]
    for col in CAT_VARS_R1:
        enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        one = enc.fit_transform(df[[col]]) / np.sqrt(df[col].nunique())
        bloques.append(one * PESOS_CATEGORICAS_R1[col])
    return np.hstack(bloques)


X_r1 = construir_matriz_r1(df_universo)
print(f'Matriz R1: {X_r1.shape[0]:,} observaciones × {X_r1.shape[1]} variables transformadas')


## Entrenamiento del modelo K-Means final

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Ajusta el modelo K-Means definitivo (k=6, 20 inicializaciones aleatorias, hasta 500 iteraciones) sobre las 18,315 observaciones del universo completo.</div>


In [ ]:
modelo_r1 = KMeans(
    n_clusters=K_R1,
    random_state=RANDOM_STATE_KM,
    n_init=20,
    max_iter=500,
)
labels_r1 = modelo_r1.fit_predict(X_r1)

print('Entrenamiento del modelo final R1 completado')
print(f'  Observaciones utilizadas: {len(labels_r1):,}')
print(f'  Variables transformadas:  {X_r1.shape[1]}')
print(f'  Clusters:                 {modelo_r1.n_clusters}')
print(f'  Iteraciones:              {modelo_r1.n_iter_}')


## Evaluación del modelo

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula las tres métricas estándar de calidad de clustering (Silhouette, Davies-Bouldin y Calinski-Harabasz) sobre el universo completo.</div>

<div style="background:#FFF8E1; border-left:6px solid #F9A825; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;"><b style="color:#B26A00;">Cómo leer estas métricas</b><ul><li><b>Silhouette</b> (entre -1 y 1, mayor es mejor): qué tan bien separado está cada punto de los clusters vecinos frente a su propio cluster. Valores &gt; 0.5 indican una estructura razonablemente clara.</li><li><b>Davies-Bouldin</b> (menor es mejor): promedio de similitud entre cada cluster y su vecino más parecido.</li><li><b>Calinski-Harabasz</b> (mayor es mejor): razón entre la dispersión entre clusters y la dispersión dentro de cada cluster.</li></ul></div>


In [ ]:
metricas_r1 = pd.DataFrame([{
    'observaciones': len(X_r1),
    'k': modelo_r1.n_clusters,
    'inercia': modelo_r1.inertia_,
    'silhouette': silhouette_score(X_r1, labels_r1),
    'davies_bouldin': davies_bouldin_score(X_r1, labels_r1),
    'calinski_harabasz': calinski_harabasz_score(X_r1, labels_r1),
}])
display(metricas_r1.style.format({
    'inercia': '{:,.2f}', 'silhouette': '{:.3f}',
    'davies_bouldin': '{:.3f}', 'calinski_harabasz': '{:,.0f}',
}))


## Visualización — proyección PCA

PCA se usa únicamente como diagnóstico visual en dos dimensiones; el modelo K-Means opera sobre el espacio completo de variables transformadas, no sobre esta proyección.

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Proyecta las observaciones y los centroides finales sobre los dos primeros componentes principales.</div>


In [ ]:
pca_r1 = PCA(n_components=2, random_state=RANDOM_STATE_KM)
X_r1_pca = pca_r1.fit_transform(X_r1)
centroides_pca = pca_r1.transform(modelo_r1.cluster_centers_)

fig, ax = plt.subplots(figsize=(10, 6))
colores_r1 = sns.color_palette('Set2', K_R1)
for g in range(K_R1):
    ix = np.flatnonzero(labels_r1 == g)
    ax.scatter(X_r1_pca[ix, 0], X_r1_pca[ix, 1], s=10, alpha=.25,
               color=colores_r1[g], label=f'Cluster {g}')
ax.scatter(centroides_pca[:, 0], centroides_pca[:, 1], marker='X', s=180,
           color='black', edgecolor='white', linewidth=1, label='Centroides')
ax.set(title='Proyección PCA del modelo final R1',
       xlabel=f'PC1 ({pca_r1.explained_variance_ratio_[0]:.1%})',
       ylabel=f'PC2 ({pca_r1.explained_variance_ratio_[1]:.1%})')
ax.legend(ncol=2)
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()


## Distribución de tamaños por cluster

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Resume cuántos leads quedaron asignados a cada cluster, en cifras absolutas y porcentaje, y confirma que ningún cluster es residual.</div>


In [ ]:
distribucion_r1 = (
    pd.Series(labels_r1, name='cluster')
      .value_counts().sort_index().rename('observaciones').to_frame()
)
distribucion_r1['porcentaje'] = distribucion_r1['observaciones'] / len(labels_r1) * 100
display(distribucion_r1.style.format({'porcentaje': '{:.2f}%'}))

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(distribucion_r1.index, distribucion_r1['observaciones'], color=colores_r1)
for bar, row in zip(bars, distribucion_r1.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{row.observaciones:,}\n({row.porcentaje:.1f}%)',
            ha='center', va='bottom', fontsize=9)
ax.set(title='Distribución de tamaños — modelo final R1', xlabel='Cluster', ylabel='Leads')
plt.tight_layout()
plt.show()

assert distribucion_r1['observaciones'].sum() == len(df_universo)
assert len(distribucion_r1) == K_R1


## Validación estadística de las diferencias entre clusters

Se contrasta si los clusters difieren de forma estadísticamente significativa en cada variable original (no solo en el espacio transformado). Para la única variable numérica (edad) se usa ANOVA de una vía con comparaciones post hoc de Tukey; para las categóricas, la prueba χ² de independencia con el tamaño de efecto V de Cramér.

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Ejecuta ANOVA + Tukey HSD sobre edad y χ² + V de Cramér sobre las cuatro variables categóricas.</div>


In [ ]:
df_final_r1 = df_universo.copy()
df_final_r1['cluster'] = labels_r1


def eta2(values, labels):
    x = np.asarray(values, float)
    total = ((x - x.mean()) ** 2).sum()
    if total == 0:
        return 0
    return sum(len(x[labels == g]) * (x[labels == g].mean() - x.mean()) ** 2
               for g in np.unique(labels)) / total


def interpretar_eta2(x):
    return 'despreciable' if x < .01 else 'pequeño' if x < .06 else 'mediano' if x <= .14 else 'grande'


def cramer_v(values, labels):
    ct = pd.crosstab(values, labels).to_numpy()
    n = ct.sum()
    ex = ct.sum(1)[:, None] * ct.sum(0)[None, :] / n
    chi = np.where(ex > 0, (ct - ex) ** 2 / ex, 0).sum()
    q = min(ct.shape) - 1
    return 0 if q == 0 else np.sqrt((chi / n) / q)


def interpretar_cramer(x):
    return 'despreciable' if x < .10 else 'débil' if x < .20 else 'moderado' if x <= .40 else 'fuerte'


# --- ANOVA + Tukey sobre edad ---
groups = [df_final_r1.loc[df_final_r1.cluster == g, 'edad'].to_numpy() for g in range(K_R1)]
F, p = f_oneway(*groups)
eta = eta2(df_final_r1['edad'], df_final_r1['cluster'])
anova_r1 = pd.DataFrame([{'variable': 'edad', 'F': F, 'p_value': p,
                          'eta_cuadrado': eta, 'magnitud_practica': interpretar_eta2(eta)}])
display(anova_r1.style.format({'F': '{:,.2f}', 'p_value': '{:.2e}', 'eta_cuadrado': '{:.3f}'}))

tukey = tukey_hsd(*groups)
sig, no_sig = [], []
for i in range(K_R1):
    for j in range(i + 1, K_R1):
        (sig if tukey.pvalue[i, j] < .05 else no_sig).append(f'{i}–{j}')
print(f'Comparaciones Tukey significativas: {len(sig)} de {len(sig) + len(no_sig)}')
if no_sig:
    print(f'Pares sin diferencia significativa: {", ".join(no_sig)}')

# --- Chi2 + V de Cramér sobre las categóricas ---
cat_rows = []
for var in CAT_VARS_R1:
    ct = pd.crosstab(df_final_r1['cluster'], df_final_r1[var])
    chi, p_val, _, _ = chi2_contingency(ct)
    q = min(ct.shape) - 1
    v = np.sqrt((chi / ct.to_numpy().sum()) / q)
    cat_rows.append({'variable': var, 'chi_cuadrado': chi, 'p_value': p_val,
                      'cramer_v': v, 'magnitud_practica': interpretar_cramer(v)})
categoricas_r1 = pd.DataFrame(cat_rows)
display(categoricas_r1.style.format({'chi_cuadrado': '{:,.1f}', 'p_value': '{:.2e}', 'cramer_v': '{:.3f}'}))


**Interpretación.** Todas las variables muestran asociaciones estadísticamente significativas con el cluster asignado (p<0.001 en todos los casos). `razon_de_compra` suele mostrar una asociación casi perfecta con el cluster (cada grupo queda compuesto casi exclusivamente por vivienda o por inversión), lo que refleja una decisión metodológica: esta variable fue ponderada explícitamente porque distingue el objetivo comercial del lead. Sexo y estado civil también muestran asociaciones fuertes, aportando el contenido demográfico de los perfiles. Edad, la única variable numérica, es el diferenciador continuo del modelo.


## Variables que más diferencian los clusters

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Ordena las cinco variables obligatorias según su tamaño de efecto (η² para edad, V de Cramér para las categóricas) para identificar qué variables impulsan más la segmentación.</div>


In [ ]:
ranking_r1 = pd.concat([
    anova_r1[['variable', 'eta_cuadrado']].rename(columns={'eta_cuadrado': 'tamano_efecto'}).assign(metrica='η²'),
    categoricas_r1[['variable', 'cramer_v']].rename(columns={'cramer_v': 'tamano_efecto'}).assign(metrica='V de Cramér'),
]).sort_values('tamano_efecto', ascending=False).reset_index(drop=True)
ranking_r1.index = ranking_r1.index + 1
ranking_r1.index.name = 'rango'
display(ranking_r1.style.format({'tamano_efecto': '{:.3f}'}).background_gradient(subset=['tamano_efecto'], cmap='YlOrRd'))

fig, ax = plt.subplots(figsize=(9, 5))
plot_rank = ranking_r1.sort_values('tamano_efecto')
ax.barh(plot_rank['variable'], plot_rank['tamano_efecto'], color='#5B8DB8')
ax.set(title='Poder discriminante de las variables — modelo final R1',
       xlabel='Tamaño de efecto', ylabel='Variable')
plt.tight_layout()
plt.savefig(OUT_DIR_KM / 'poder_discriminante_R1.png', dpi=300, bbox_inches='tight')
plt.show()


## Perfil de cada cluster

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula el tamaño, la edad promedio y la categoría dominante de cada variable en cada cluster, y genera automáticamente un nombre descriptivo a partir de esas categorías dominantes.</div>


In [ ]:
def moda(s):
    return s.mode().iloc[0]


resumen_clusters_r1 = df_final_r1.groupby('cluster').agg(
    observaciones=('cluster', 'size'),
    tasa_conversion=('target_won', 'mean'),
    edad_promedio=('edad', 'mean'),
    razon_dominante=('razon_de_compra', moda),
    tipo_dominante=('tipo_unidad', moda),
    estado_civil_dominante=('estado_civil', moda),
    sexo_dominante=('sexo', moda),
).reset_index()
resumen_clusters_r1['porcentaje_dataset'] = resumen_clusters_r1['observaciones'] / len(df_final_r1) * 100


def nombrar_cluster(row):
    base = 'Inversionistas' if row['razon_dominante'] == 'Inversión' else f"Compradores de vivienda ({row['tipo_dominante']})"
    civil_txt = 'casados' if 'asad' in str(row['estado_civil_dominante']) else 'solteros'
    sexo_txt = {'Masculino': 'hombres', 'Femenino': 'mujeres'}.get(row['sexo_dominante'], row['sexo_dominante'])
    return f"{base}, {civil_txt} — {sexo_txt}, edad prom. {row['edad_promedio']:.0f}"


resumen_clusters_r1['nombre'] = resumen_clusters_r1.apply(nombrar_cluster, axis=1)
cols = ['cluster', 'nombre', 'observaciones', 'porcentaje_dataset', 'tasa_conversion',
        'edad_promedio', 'razon_dominante', 'tipo_dominante', 'estado_civil_dominante', 'sexo_dominante']
display(resumen_clusters_r1[cols].style.format({
    'porcentaje_dataset': '{:.1f}%', 'tasa_conversion': '{:.2%}', 'edad_promedio': '{:.1f}',
}).background_gradient(subset=['tasa_conversion'], cmap='YlGn'))


**Cómo leer esta tabla.** Cada fila es uno de los seis clusters del modelo final. `tasa_conversion` es la proporción de leads de ese cluster que efectivamente cerraron una compra (`target_won=1`); esta variable **no participó en la formación de los clusters** — se calcula después, únicamente como validación externa de si la segmentación tiene relación con el resultado de negocio. Los nombres se generan automáticamente a partir de la categoría dominante de cada variable en el cluster; en la práctica, el modelo separa principalmente por **motivación de compra** (vivienda vs. inversión), y dentro de cada motivación, por **sexo**, **estado civil** y **edad**.


## Conversión por cluster

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Para cada cluster calcula la tasa de conversión, un intervalo de confianza de Wilson al 95%, el lift respecto a la tasa global y una prueba de dos proporciones contra el resto del universo — para saber si la diferencia observada es estadísticamente significativa o podría deberse al azar.</div>


In [ ]:
global_conversion_r1 = df_final_r1['target_won'].mean()
conv_rows = []
for g, x in df_final_r1.groupby('cluster'):
    n = len(x)
    wins = int(x['target_won'].sum())
    ph = wins / n
    z = 1.96
    den = 1 + z * z / n
    center = (ph + z * z / (2 * n)) / den
    half = z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / den
    rest = df_final_r1.loc[df_final_r1.cluster != g, 'target_won']
    pooled = (wins + rest.sum()) / (n + len(rest))
    se = np.sqrt(pooled * (1 - pooled) * (1 / n + 1 / len(rest)))
    p_vs = 2 * norm.sf(abs((ph - rest.mean()) / se))
    conv_rows.append({
        'cluster': g, 'observaciones': n, 'convertidos': wins, 'tasa_conversion': ph,
        'IC95_inferior': center - half, 'IC95_superior': center + half,
        'lift': ph / global_conversion_r1, 'p_vs_resto': p_vs,
        'clasificacion': ('superior' if ph > rest.mean() and p_vs < .05
                           else 'inferior' if ph < rest.mean() and p_vs < .05 else 'sin diferencia'),
    })
conversion_r1 = pd.DataFrame(conv_rows)
display(conversion_r1.style.format({
    'tasa_conversion': '{:.2%}', 'IC95_inferior': '{:.2%}', 'IC95_superior': '{:.2%}',
    'lift': '{:.2f}×', 'p_vs_resto': '{:.2e}',
}).background_gradient(subset=['lift'], cmap='YlGn'))

fig, ax = plt.subplots(figsize=(9, 5))
rates = conversion_r1['tasa_conversion'] * 100
err = np.vstack([
    (conversion_r1['tasa_conversion'] - conversion_r1['IC95_inferior']) * 100,
    (conversion_r1['IC95_superior'] - conversion_r1['tasa_conversion']) * 100,
])
bars = ax.bar(conversion_r1['cluster'], rates, yerr=err, capsize=5, color=colores_r1, edgecolor='white')
ax.axhline(global_conversion_r1 * 100, color='red', ls='--', label=f'Conversión global: {global_conversion_r1:.2%}')
for b, v, lift in zip(bars, rates, conversion_r1['lift']):
    ax.text(b.get_x() + b.get_width() / 2, v + .25, f'{v:.2f}%\n{lift:.2f}×', ha='center', fontsize=9)
ax.set(title='Conversión e intervalos de confianza del 95% por cluster — modelo final R1',
       xlabel='Cluster', ylabel='Conversión (%)')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR_KM / 'conversion_clusters_R1.png', dpi=300, bbox_inches='tight')
plt.show()


## Separación entre clusters

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Calcula la distancia euclidiana entre los centroides finales, en el mismo espacio de variables transformadas usado para entrenar el modelo, para identificar qué pares de clusters son más parecidos y cuáles más distintos entre sí.</div>


In [ ]:
dist_centroides_r1 = pairwise_distances(modelo_r1.cluster_centers_)
tabla_dist_r1 = pd.DataFrame(dist_centroides_r1,
                              index=[f'C{i}' for i in range(K_R1)],
                              columns=[f'C{i}' for i in range(K_R1)])
tri = np.triu_indices(K_R1, 1)
values = dist_centroides_r1[tri]
imin, imax = values.argmin(), values.argmax()
resumen_dist_r1 = pd.DataFrame([{
    'distancia_promedio': values.mean(), 'distancia_minima': values.min(),
    'clusters_mas_similares': f'C{tri[0][imin]}–C{tri[1][imin]}',
    'distancia_maxima': values.max(),
    'clusters_mas_diferentes': f'C{tri[0][imax]}–C{tri[1][imax]}',
}])
display(resumen_dist_r1.style.format({
    'distancia_promedio': '{:.3f}', 'distancia_minima': '{:.3f}', 'distancia_maxima': '{:.3f}',
}))

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(tabla_dist_r1, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax, square=True)
ax.set_title('Distancia euclidiana entre centroides — modelo final R1')
plt.tight_layout()
plt.savefig(OUT_DIR_KM / 'distancias_centroides_R1.png', dpi=300, bbox_inches='tight')
plt.show()


## Interpretación de negocio e implicaciones para la empresa

- **Prioridad comercial alta — clusters de inversión.** Los segmentos donde predomina la motivación de inversión suelen mostrar conversión significativamente superior al promedio (ver tabla de conversión anterior). Los argumentos comerciales deben centrarse en la motivación de inversión declarada; cualquier promesa de rentabilidad requeriría información financiera adicional no disponible en esta base.
- **Vivienda, casados, tres habitaciones.** Los clusters de compradores casados orientados a vivienda de tres habitaciones suelen convertir por debajo del promedio. Antes de intensificar el contacto conviene revisar disponibilidad, adecuación de producto y objeciones asociadas a esa tipología. Sexo se usa para describir el perfil, no como criterio de exclusión.
- **Vivienda, jóvenes, solteros.** Los clusters más jóvenes y solteros orientados a vivienda de dos habitaciones muestran conversión variable; ameritan seguimiento estándar o revisión temprana de objeciones, según el cluster específico.
- **Marketing.** La bifurcación más accionable es motivación (vivienda vs. inversión); dentro de cada una, los mensajes pueden adaptarse por estado civil, edad y tipo de unidad preferido.
- **Priorización operativa.** El lift de conversión de la tabla anterior permite ordenar la atención comercial entre clusters, sujeto siempre a capacidad operativa y sin reemplazar la evaluación individual de cada lead.

## Limitaciones

- El universo final incluye 18,315 de 42,414 leads calificados, porque exige completitud en las cinco variables obligatorias. Esta selección puede introducir sesgo frente a leads con información demográfica incompleta.
- El modelo no incorpora variables económicas (ingreso, precio, m²) en su formación ni en su perfilamiento descriptivo — una decisión metodológica deliberada para obtener una lectura estrictamente demográfico-conductual y de producto, no económica.
- El CRM no dispone con cobertura suficiente de hijos, dependientes, ocupación, estabilidad laboral, capacidad de cuota ni comportamiento detallado de seguimiento; estas ausencias limitan la riqueza del perfil.
- K-Means presupone distancias euclidianas, centroides y grupos aproximadamente compactos; las ponderaciones de R1 reflejan una decisión metodológica explícita, no un óptimo matemático único.
- `razon_de_compra` puede quedar casi determinada por el cluster (asociación muy fuerte), lo que mejora la separación pero introduce un componente casi circular que debe declararse al interpretar el modelo.
- La conversión se mide después del clustering y no demuestra causalidad; las diferencias observadas pueden reflejar variables omitidas o condiciones comerciales no registradas.


## Exportación del modelo final y resultados

<div style="color:#164e63; background-color:#ecfeff; border-left:4px solid #0891b2; padding:10px 12px; margin:8px 0 10px 0; border-radius:4px;"><strong>Qué hace este chunk:</strong> Guarda las etiquetas de cluster, las tablas de perfilamiento, las validaciones estadísticas y el modelo entrenado, para su uso fuera del notebook.</div>


In [ ]:
import joblib

resumen_clusters_r1.to_csv(OUT_DIR_KM / 'resumen_clusters_R1.csv', index=False, encoding='utf-8-sig')
anova_r1.to_csv(OUT_DIR_KM / 'anova_eta2_R1.csv', index=False, encoding='utf-8-sig')
categoricas_r1.to_csv(OUT_DIR_KM / 'chi2_cramer_R1.csv', index=False, encoding='utf-8-sig')
ranking_r1.to_csv(OUT_DIR_KM / 'ranking_variables_R1.csv', encoding='utf-8-sig')
conversion_r1.to_csv(OUT_DIR_KM / 'conversion_intervalos_lift_R1.csv', index=False, encoding='utf-8-sig')
tabla_dist_r1.to_csv(OUT_DIR_KM / 'distancias_centroides_R1.csv', encoding='utf-8-sig')
metricas_r1.to_csv(OUT_DIR_KM / 'metricas_R1.csv', index=False, encoding='utf-8-sig')
distribucion_r1.to_csv(OUT_DIR_KM / 'distribucion_R1.csv', encoding='utf-8-sig')
pd.DataFrame({'cluster_R1': labels_r1}, index=df_universo.index).to_csv(
    OUT_DIR_KM / 'asignacion_clusters_R1.csv', index_label='indice_original', encoding='utf-8-sig')
joblib.dump(modelo_r1, OUT_DIR_KM / 'modelo_KMeans_R1.joblib')

print(f'Resultados del modelo final R1 exportados en: {OUT_DIR_KM}')


---
# Parte III — Modelo Predictivo de Conversión (Regresión Logística)

Esta parte reproduce el **modelo predictivo final** obtenido en `03_modelado_predictivo_leads_qualified.ipynb` (carpeta `03 Prediccion/`), un notebook de trabajo de más de 250 celdas que exploró numerosos algoritmos, conjuntos de variables, estrategias de balanceo y universos de entrenamiento antes de llegar al modelo seleccionado.

El código de esta parte se copia, salvo indicación explícita, **tal como aparece en el notebook original** — incluyendo los recuadros de explicación (azul: *qué se hizo*; morado: *qué hace el código*; ámbar: *cómo leer*) que ya fueron validados como claros para una audiencia no técnica.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, time, pickle, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, HistGradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import shap
# shap.initjs()  # browser-only, skipped in nbconvert

# ── Reproducibility seed ──────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Output directories ────────────────────────────────────────────────────
BASE_DIR   = Path(f'{REPO_ROOT}/03 Prediccion')
RUN_DIR    = BASE_DIR / 'qualified_run'
DATA_PATH  = Path(f'{REPO_ROOT}/01 Limpieza Código/data/features/leads_enriched_qualified_final.csv')
KMEANS_PATH= Path(f'{REPO_ROOT}/02 K Means/Output/leads_calificados_clustered.csv')
FIGURES_DIR= RUN_DIR / 'figures'
MODELS_DIR = RUN_DIR / 'models'
RESULTS_DIR= RUN_DIR / 'results'
for d in [FIGURES_DIR, MODELS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams['figure.dpi'] = 110
matplotlib.rcParams['font.size']   = 10
import sklearn
print('Librerias cargadas correctamente:')
print(f'  sklearn  : {sklearn.__version__}')
print(f'  imbalanced-learn: {__import__("imblearn").__version__}')
print(f'  shap     : {shap.__version__}')
print()
print('NOTA: XGBoost y LightGBM requieren libomp ARM64 que no esta disponible en')
print('este sistema. Se sustituyen por:')
print('  - HistGradientBoostingClassifier (sklearn) como alternativa a LightGBM')
from sklearn.inspection import permutation_importance
print('  - ExtraTreesClassifier (sklearn) como alternativa a XGBoost')
print('  Ambos son mencionados explicitamente en los requisitos del estudio.')


In [ ]:
DATA_PATH = Path(f'{REPO_ROOT}/01 Limpieza Código/'
                 'data/features/leads_enriched_qualified_final.csv')

df_model = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Dataset principal: {df_model.shape[0]:,} filas x {df_model.shape[1]} columnas")
print(f"Conversiones (target_won=1): {df_model['target_won'].sum():,} "
      f"({df_model['target_won'].mean()*100:.2f}%)")


In [ ]:
# Punto medio de los rangos declarados de ingreso familiar (en miles de Q),
# usado como variable numérica y en las interacciones edad×ingreso, ingreso×inversión.
INCOME_MID = {'Menos De Q10k':8,'Q 10k A Q13.9k':12,'Q 14k A Q17.9k':16,'Q18k A Q21.9k':20,
 'Q22k A Q25.9k':24,'Q26k A Q29.9k':28,'Q30k A 33.9k':32,'Q34k A 37.9k':36,'Q38k A 41.9k':40,
 'Q42k A 45.9k':44,'46k A 49.9k':48,'Q50k A Más':55}
df_model['ingreso_mid'] = df_model['ingreso_familiar'].map(INCOME_MID)
df_model['edad'] = pd.to_numeric(df_model['edad'], errors='coerce')


In [ ]:
# Función auxiliar reutilizada más adelante: métricas de negocio "al top K%"
# (precisión, recall y lift si se contacta solo al K% de leads mejor rankeado).
def business_at_k(y_true, y_prob, k_list=(0.05, 0.10, 0.20, 0.30)):
    n = len(y_true); tot = y_true.sum(); order = np.argsort(y_prob)[::-1]
    cum = np.cumsum(np.asarray(y_true)[order]); out = {}
    for k in k_list:
        tn = max(1, int(np.ceil(n * k))); tp = cum[tn - 1]
        out[k] = {'n': tn, 'tp': int(tp), 'prec': tp / tn, 'rec': tp / tot if tot else 0.0,
                  'lift': (tp / tn) / (tot / n) if tot else 0.0}
    return out


## Búsqueda exhaustiva de configuraciones (resumen)

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (resumen).</b><br><br>
Antes de llegar al modelo final se realizó una <b>búsqueda exhaustiva</b> ejecutada con scripts externos (guardados en <code>qualified_run/exploracion/</code>): (1) un <i>grid</i> amplio de ~100 configuraciones (6 conjuntos de variables × 9 algoritmos × 5 estrategias de balanceo, evaluadas con validación cruzada 5-fold *out-of-fold*); (2) ajuste de hiperparámetros sobre los mejores candidatos; (3) selección de variables por tres métodos independientes (univariado, <i>forward</i> y <i>backward</i>); (4) una ablación por familias de variables para medir cuánto aporta cada bloque de información; y (5) una reconstrucción del universo de entrenamiento probando ocho variantes de completitud de datos.
<br><br>
El resultado ganador es el <b>Universo H</b> — leads con canal, edad, ingreso y fuente completos (22,656 leads, 3.26% de conversión) — junto con 24 variables predictoras (incluyendo ingeniería de interacciones) y una <b>Regresión Logística con regularización L2 (C=0.55)</b>. La familia de variables de <b>canal de adquisición</b> resultó ser, por un margen amplio, la más determinante; edad e ingreso aportan señal adicional a través de sus interacciones. Las celdas siguientes reconstruyen ese modelo final de forma autocontenida y muestran todas sus métricas de evaluación.
</div>


### Sección 21.4 — Modelo final (reproducible en el notebook)

**Universo H:** leads con `fuente_pt2`, `canal_de_anuencia`, `canal_de_contacto`, `ingreso_familiar` y `edad` completas (22,656 leads, 739 conversiones, tasa 3.26%). Las demás variables entran como opcionales (categoría `_missing_` cuando faltan — señal de completitud de perfil, metodología aceptada desde la Sección 14; las variables de fuga de la Sección 20 siguen excluidas).

**Modelo:** Regresión Logística L2, `C=0.55` (tuneada en fase 1b), OHE con agrupación de categorías raras (`min_frequency=30`) — este agrupamiento fue clave para destapar la señal de canal que el hold-out simple de la Sección 19 no veía.

**Generalización a proyectos nuevos (decisión final):** `fuente_pt2` y `proyecto` —la identidad del proyecto— quedan **fuera de las variables predictoras**, porque el objetivo del modelo es aplicarse a proyectos futuros que no existen en los datos de entrenamiento. `fuente_pt2` se mantiene únicamente como requisito de completitud del universo H. Costo medido de excluirlas (OOF 5-fold, 2 semillas, mismas filas): PR-AUC 0.221 → 0.210, lift@10% 4.3 → 4.0–4.1, y el punto de recall 0.70 pasa de 26.6–29.0% a 29.0–30.0% de leads — un costo pequeño que se acepta a cambio de un modelo **portable a proyectos nuevos**.

### ¿Qué hace el código de la celda siguiente? (§21.4)

<div style="background:#EDE7F6; border-left:6px solid #6A1B9A; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#6A1B9A;">Qué hace el código</b><br><br>
<b>En palabras simples:</b> esta celda construye el modelo final y lo evalúa de forma honesta — cada lead se califica con un modelo que <b>nunca lo vio</b> durante el entrenamiento.
<ol>
<li><b>Crea variables derivadas:</b> agrupa los canales de anuencia en familias (<code>pauta_digital</code>, <code>portal</code>, <code>directo</code>, <code>referido</code>, <code>presencial</code>, …), marca si el lead es referido o compra por inversión, y construye interacciones (p. ej. <code>edad × ingreso</code>) que capturan combinaciones que una variable sola no muestra.</li>
<li><b>Define el universo H:</b> se queda solo con los leads que tienen completos fuente, canal de anuencia, canal de contacto, ingreso y edad (22,656 leads). Las demás variables pueden faltar: el faltante se convierte en la categoría <code>_missing_</code>. Ojo: la fuente y el proyecto solo se usan para definir el universo, <b>no como variables predictoras</b> — así el modelo generaliza a proyectos nuevos.</li>
<li><b>Arma el pipeline:</b> imputación de faltantes + one-hot encoding (agrupando categorías con menos de 30 casos) + escalado de numéricas + Regresión Logística con regularización L2 (<code>C=0.55</code>).</li>
<li><b>Evalúa con validación cruzada OOF (out-of-fold):</b> divide los datos en 5 partes; cada parte se predice con un modelo entrenado en las otras 4. Se repite con 2 semillas distintas para confirmar que el resultado no depende del azar de la partición.</li>
<li><b>Construye la tabla de umbrales operativos:</b> para cada recall objetivo (50%, 60%, 70%, 80%) calcula qué umbral de probabilidad hay que usar, cuántos leads implica contactar y con qué precisión.</li>
</ol>
</div>

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Resultados clave para las secciones siguientes:</b> <code>prob_H</code> (probabilidad OOF de cada lead), <code>y_H</code> (si convirtió o no) y <code>thr_final</code> (tabla de umbrales operativos).
</div>


In [ ]:
# ── §21.4  Modelo final: universo H + LR tuneada, OOF con 2 semillas ─────
from sklearn.model_selection import cross_val_predict

CANAL_FAMILIA = {
 'Facebook':'pauta_digital','Instagram':'pauta_digital','Fanpage Líderes':'pauta_digital',
 'Inbox':'pauta_digital','portal_demo':'portal','Página Web':'portal',
 'WhatsApp':'directo','Llamada':'directo',
 'Referencia':'referido','Friends/Family':'referido','Shareholders Sale':'referido',
 'Sede':'presencial','ExpoCasa':'presencial','ExpoCasa Virtual':'presencial',
 'Visita Inédita':'presencial','Open House':'presencial','OpenSale':'presencial',
 'Ferias':'presencial','Evento':'presencial','Actividades MKT':'presencial',
 'Actividad Posicionamiento':'presencial','Lanzamiento':'presencial',
 'Vallas':'exterior','Corporación':'otro','usa':'otro','Prensa':'otro','Radio/tv':'otro'}

df_f = df_model.copy()
df_f['edad_n'] = pd.to_numeric(df_f['edad'], errors='coerce')
df_f['es_inversion'] = (df_f['razon_de_compra']=='Inversión').astype(int)
df_f['canal_familia'] = df_f['canal_de_anuencia'].map(CANAL_FAMILIA).fillna('otro')
df_f['es_referido'] = (df_f['canal_familia']=='referido').astype(int)
df_f['edad_x_inversion'] = df_f['edad_n']*df_f['es_inversion']
df_f['edad_x_ingreso'] = df_f['edad_n']*df_f['ingreso_mid']/100.0
df_f['edad_x_referido'] = df_f['edad_n']*df_f['es_referido']
df_f['mayor_45'] = (df_f['edad_n']>=45).astype(int)
df_f['ingreso_x_inversion'] = df_f['ingreso_mid']*df_f['es_inversion']
df_f['ingreso_alto'] = (df_f['ingreso_mid']>=32).astype(int)

REQ_H = ['fuente_pt2','canal_de_anuencia','canal_de_contacto','ingreso_familiar','edad']
# fuente_pt2 y proyecto (identidad de proyecto) se EXCLUYEN como predictoras:
# el objetivo es generalizar a proyectos nuevos. fuente_pt2 permanece en REQ_H
# solo como requisito de completitud del universo, no como variable del modelo.
FEATS_H = ['edad_n','sexo','estado_civil','ingreso_mid','ingreso_familiar','razon_de_compra',
 'canal_de_anuencia','canal_de_contacto','canal_familia','tipo_unidad','opciones_visitadas',
 'participantes_en_credito','lugar_trabajo','lugar_vivienda','campania_tipo','campania_audiencia',
 'es_inversion','es_referido','edad_x_inversion','ingreso_x_inversion','edad_x_ingreso',
 'edad_x_referido','mayor_45','ingreso_alto']

mask_H = df_f[REQ_H].notna().all(axis=1)
df_H = df_f.loc[mask_H].reset_index(drop=True)
y_H = df_H['target_won'].values
print(f"Universo H: {len(df_H):,} leads | {int(y_H.sum())} conversiones ({y_H.mean()*100:.2f}%)")

cat_H = [c for c in FEATS_H if not pd.api.types.is_numeric_dtype(df_H[c])]
num_H = [c for c in FEATS_H if c not in cat_H]
prep_H = ColumnTransformer([
  ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='_missing_')),
                    ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                                          min_frequency=30))]), cat_H),
  ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                    ('sc', StandardScaler())]), num_H)], remainder='drop')
pipe_H = Pipeline([('prep', prep_H),
                   ('clf', LogisticRegression(C=0.55, penalty='l2', max_iter=2000,
                                              random_state=SEED))])

met_rows=[]
prob_H = None
for tag,seed in [('selección',SEED),('confirmación',2025)]:
    skf_f = StratifiedKFold(5, shuffle=True, random_state=seed)
    prob = cross_val_predict(pipe_H, df_H[FEATS_H], y_H, cv=skf_f,
                             method='predict_proba', n_jobs=-1)[:,1]
    if prob_H is None: prob_H = prob
    bm = business_at_k(y_H, prob)
    prc,rcc,thr = precision_recall_curve(y_H, prob)
    i70 = np.where(rcc[:-1]>=0.70)[0][-1]
    met_rows.append({'CV':tag,'pr_auc':round(average_precision_score(y_H,prob),4),
        'roc_auc':round(roc_auc_score(y_H,prob),4),
        'lift@10%':round(bm[0.10]['lift'],2),'recall@20%':round(bm[0.20]['rec'],3),
        'pct_leads_r70':round((prob>=thr[i70]).mean()*100,1),
        'prec_r70':round(prc[i70],4)})
met_final = pd.DataFrame(met_rows)
print("\nMÉTRICAS DEL MODELO FINAL (OOF 5-fold, dos semillas):")
print(met_final.to_string(index=False))
met_final.to_csv(RESULTS_DIR/'modelo_final_metricas.csv', index=False)

thr_rows=[]
prc,rcc,thr = precision_recall_curve(y_H, prob_H)
for target in [0.50,0.60,0.70,0.80]:
    idx=np.where(rcc[:-1]>=target)[0]
    if len(idx):
        i=idx[-1]; sel=prob_H>=thr[i]
        thr_rows.append({'recall_objetivo':target,'umbral':round(thr[i],4),
            'recall':round(rcc[i],3),'precision':round(prc[i],4),
            'lift_vs_base':round(prc[i]/y_H.mean(),2),
            'leads':int(sel.sum()),'pct_leads':round(sel.mean()*100,1)})
thr_final = pd.DataFrame(thr_rows)
print("\nUMBRALES OPERATIVOS (OOF selección):")
print(thr_final.to_string(index=False))
thr_final.to_csv(RESULTS_DIR/'modelo_final_umbrales.csv', index=False)

### Sección 21.5 — Curvas del modelo final

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo</b><br><br>
Se grafican cuatro vistas complementarias del desempeño del modelo final, usando las probabilidades OOF (cada lead evaluado por un modelo que no lo vio al entrenar).
</div>

<div style="background:#EDE7F6; border-left:6px solid #6A1B9A; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#6A1B9A;">Qué hace el código</b><br><br>
Ordena los leads de mayor a menor probabilidad y acumula las conversiones capturadas (curvas de ganancia y lift); calcula la curva precisión-recall con <code>precision_recall_curve</code> y la curva de calibración con <code>calibration_curve</code> (10 grupos de igual tamaño).
</div>

<div style="background:#FFF8E1; border-left:6px solid #F9A825; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#B26A00;">Cómo leer cada gráfico</b>
<table style="border-collapse:collapse; margin-top:8px;">
<tr><th style="border:1px solid #D9C46B; padding:6px 10px; background:#FCF0C8; text-align:left; color:#212121;">Gráfico</th><th style="border:1px solid #D9C46B; padding:6px 10px; background:#FCF0C8; text-align:left; color:#212121;">Cómo leerlo</th></tr>
<tr><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;"><b>Ganancia acumulada</b> (arriba izq.)</td><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;">Eje X: % de leads contactados, empezando por los de mayor probabilidad. Eje Y: % de las conversiones reales capturadas. Cuanto más se despega la curva azul de la diagonal ("aleatorio"), mejor. La línea roja punteada marca la meta de capturar el 70%.</td></tr>
<tr><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;"><b>Lift</b> (arriba der.)</td><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;">Cuántas veces más conversiones se obtienen respecto a contactar al azar. Un lift de 4 en el 10% significa: contactando solo al 10% mejor rankeado, se consiguen 4× más conversiones que eligiendo leads al azar.</td></tr>
<tr><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;"><b>Precisión-Recall</b> (abajo izq.)</td><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;">El intercambio entre capturar más conversiones (recall, eje X) y acertar más entre los contactados (precisión, eje Y). La línea gris es la tasa base (3.26%): mientras la curva esté por encima, el modelo aporta valor sobre no usar modelo.</td></tr>
<tr><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;"><b>Calibración</b> (abajo der.)</td><td style="border:1px solid #D9C46B; padding:6px 10px; color:#212121;">Compara la probabilidad predicha con la frecuencia real de conversión. Si los puntos siguen la diagonal, las probabilidades son confiables: un lead con probabilidad 0.10 realmente convierte ≈10% de las veces.</td></tr>
</table>
</div>


In [ ]:
# ── §21.5  Curvas del modelo final ────────────────────────────────────────
from sklearn.calibration import calibration_curve
fig,ax=plt.subplots(2,2,figsize=(14,10))
order=np.argsort(prob_H)[::-1]; cg=np.cumsum(y_H[order])/y_H.sum()
pct=np.arange(1,len(y_H)+1)/len(y_H)
ax[0,0].plot(pct,cg,lw=2,color='#1565C0',label='Modelo final')
ax[0,0].plot([0,1],[0,1],'k--',lw=.8,label='Aleatorio')
ax[0,0].axhline(0.70,color='red',ls=':',lw=1,label='Recall 70%')
ax[0,0].set_title('Curva de ganancia acumulada',fontweight='bold')
ax[0,0].set_xlabel('% leads contactados');ax[0,0].set_ylabel('% conversiones capturadas');ax[0,0].legend(fontsize=8)
lift=cg/np.clip(pct,1e-9,None)
ax[0,1].plot(pct,lift,lw=2,color='#6A1B9A');ax[0,1].axhline(1,color='k',ls='--',lw=.8)
ax[0,1].set_title('Curva de lift',fontweight='bold')
ax[0,1].set_xlabel('% leads contactados');ax[0,1].set_ylabel('Lift');ax[0,1].set_xlim(0,1)
prc,rcc,_=precision_recall_curve(y_H,prob_H)
ax[1,0].plot(rcc,prc,lw=2,color='#2E7D32',label=f'AP={average_precision_score(y_H,prob_H):.3f}')
ax[1,0].axhline(y_H.mean(),color='gray',ls='--',lw=.8,label=f'Base={y_H.mean():.3f}')
ax[1,0].set_title('Curva Precision-Recall',fontweight='bold')
ax[1,0].set_xlabel('Recall');ax[1,0].set_ylabel('Precision');ax[1,0].legend(fontsize=8)
fp,mpv=calibration_curve(y_H,prob_H,n_bins=10,strategy='quantile')
ax[1,1].plot(mpv,fp,'o-',color='#C62828',label='Modelo')
ax[1,1].plot([0,max(mpv)],[0,max(mpv)],'k--',lw=.8,label='Perfecta')
ax[1,1].set_title('Curva de calibración',fontweight='bold')
ax[1,1].set_xlabel('Prob. predicha media');ax[1,1].set_ylabel('Frac. positivos observada');ax[1,1].legend(fontsize=8)
plt.suptitle('Modelo final — LR (C=0.55) | Universo H (20,304 leads) | OOF 5-fold',
             fontsize=13,fontweight='bold')
plt.tight_layout();plt.savefig(FIGURES_DIR/'modelo_final_curvas.png',dpi=140,bbox_inches='tight');plt.show()

### Sección 21.6 — Importancia de variables y serialización del modelo

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo</b><br><br>
Se midió qué variables sostienen al modelo final con dos métodos independientes, y se guardó el modelo entrenado (<code>.pkl</code>) para poder usarlo fuera del notebook.
</div>

<div style="background:#EDE7F6; border-left:6px solid #6A1B9A; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#6A1B9A;">Qué hace el código</b>
<ol>
<li>Separa un 20% de hold-out y entrena el pipeline con el 80% restante.</li>
<li><b>Permutation importance:</b> desordena una variable a la vez y mide cuánto cae el PR-AUC. Si al desordenarla el modelo empeora mucho, esa variable era importante. Se repite 15 veces para estimar la variabilidad (barras de error).</li>
<li><b>SHAP:</b> calcula la contribución media de cada <i>feature transformada</i> (cada categoría del one-hot por separado) a las predicciones individuales.</li>
<li>Guarda el pipeline completo con <code>pickle</code>, junto con la lista de features, los requisitos del universo y los umbrales operativos.</li>
</ol>
</div>

<div style="background:#FFF8E1; border-left:6px solid #F9A825; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#B26A00;">Cómo leer las gráficas</b>
<ul>
<li><b>Izquierda (permutation):</b> barras más largas = variables más importantes, a nivel de variable original. La barra de error indica estabilidad: si es pequeña respecto a la barra, la importancia es confiable.</li>
<li><b>Derecha (SHAP):</b> la misma idea pero a nivel de categoría individual (p. ej., <code>canal_de_anuencia_Referencia</code> por separado). Permite ver <i>qué categoría específica</i> de una variable es la que mueve al modelo.</li>
</ul>
</div>


In [ ]:
# ── §21.6  Importancia (permutación + SHAP) y serialización ──────────────
import pickle
tr_H, te_H = train_test_split(np.arange(len(df_H)), test_size=0.20,
                              random_state=SEED, stratify=y_H)
pipe_H.fit(df_H.iloc[tr_H][FEATS_H], y_H[tr_H])
pi_H = permutation_importance(pipe_H, df_H.iloc[te_H][FEATS_H], y_H[te_H],
                              scoring='average_precision', n_repeats=15,
                              random_state=SEED, n_jobs=-1)
imp_H = pd.DataFrame({'variable':FEATS_H,'imp_media':pi_H.importances_mean,
                      'imp_std':pi_H.importances_std}).sort_values('imp_media',ascending=False)
print("PERMUTATION IMPORTANCE (hold-out 20% del universo H) — top 15:")
print(imp_H.head(15).round(4).to_string(index=False))
imp_H.to_csv(RESULTS_DIR/'modelo_final_importancia.csv', index=False)

import shap
Xt_H = pipe_H.named_steps['prep'].transform(df_H.iloc[te_H][FEATS_H])
names_H = pipe_H.named_steps['prep'].get_feature_names_out()
expl = shap.LinearExplainer(pipe_H.named_steps['clf'], Xt_H)
sv = expl.shap_values(Xt_H)
shap_H = pd.DataFrame({'feature':names_H,'mean_abs_shap':np.abs(sv).mean(axis=0)})\
    .sort_values('mean_abs_shap',ascending=False)
print("\nSHAP |valor| medio — top 15 features transformadas:")
print(shap_H.head(15).round(4).to_string(index=False))
shap_H.to_csv(RESULTS_DIR/'modelo_final_shap.csv', index=False)

fig,ax=plt.subplots(1,2,figsize=(14,6))
t=imp_H.head(14).iloc[::-1]
ax[0].barh(t['variable'],t['imp_media'],xerr=t['imp_std'],color='#00897B',alpha=.88,capsize=3)
ax[0].set_title('Permutation importance (± std)',fontweight='bold')
ax[0].set_xlabel('Caída de PR-AUC')
s=shap_H.head(14).iloc[::-1]
ax[1].barh(s['feature'],s['mean_abs_shap'],color='#5C6BC0',alpha=.88)
ax[1].set_title('SHAP |valor| medio (features transformadas)',fontweight='bold')
ax[1].set_xlabel('|SHAP| medio')
plt.tight_layout();plt.savefig(FIGURES_DIR/'modelo_final_importancia.png',dpi=140,bbox_inches='tight');plt.show()

with open(MODELS_DIR/'modelo_final_busqueda.pkl','wb') as f:
    pickle.dump({'pipeline':pipe_H,'features':FEATS_H,'requisitos_universo':REQ_H,
                 'metricas':met_final.to_dict('records'),
                 'umbrales':thr_final.to_dict('records')}, f)
print(f"\nModelo serializado en {MODELS_DIR/'modelo_final_busqueda.pkl'}")

## Conclusión — Sección 21

**La búsqueda exhaustiva encontró un modelo sustancialmente mejor que todos los anteriores, sin variables de fuga y sin identidad de proyecto (generalizable a proyectos nuevos):**

| | Sección 20 (honesto, hold-out) | **Sección 21 (búsqueda + CV, 24 vars)** |
|---|---|---|
| Universo | casos completos (~13,038 leads, extracto preliminar) | **22,656 leads (Universo H)** |
| PR-AUC | ~0.07 (hold-out) | **0.210 (OOF, 2 semillas)** |
| % leads para recall≥0.70 | 64.2% | **29.0–30.0%** |
| Precisión en ese punto | 5.1% (1.2× base) | **7.2–7.4% (~2.4× base)** |
| Lift@10% | ~2.6 | **4.0–4.1** |

**Generalización a proyectos nuevos.** `fuente_pt2` y `proyecto` se excluyeron de las variables predictoras: la identidad del proyecto no existe para proyectos futuros, así que un modelo que dependa de ella no es aplicable donde más se necesita. El costo medido fue pequeño (PR-AUC 0.221 → 0.210; lift@10% 4.3 → 4.0–4.1; recall 0.70 pasa de costar 26.6–29.0% a 29.0–30.0% de los leads) y se acepta a cambio de la portabilidad. La búsqueda ya lo anticipaba: la permutation importance de ambas era ~0 individualmente (información duplicada entre sí).

**Por qué la Sección 19–20 subestimó la señal.** Tres causas identificadas: (1) el hold-out simple de 20% con 113 conversiones es muy ruidoso — la CV 5-fold out-of-fold usa las 625; (2) sin agrupación de categorías raras (`min_frequency`), las decenas de niveles poco frecuentes de canal/lugar diluían la señal en la regularización; (3) faltaban las interacciones (`edad×ingreso`, `edad×referido`) que portan la señal conjunta.

**Variables que realmente importan** (permutation + SHAP + ablación coinciden): la familia **canal** (`es_referido`, `canal_de_contacto`, `canal_de_anuencia`) es la columna vertebral; le siguen las **interacciones de edad** (`edad×ingreso`, `edad×inversión`) y aportes menores de `lugar_trabajo` y `campania_tipo`. Demografía pura (sexo, estado civil, hijos, núcleo) no aporta sobre lo anterior — por eso pudieron salir de los requisitos del universo.

**Universo final (H).** Exigir completas solo `fuente`, `canal_de_anuencia`, `canal_de_contacto`, `ingreso_familiar` y `edad` da 22,656 leads (50% del calificado) reteniendo 739 conversiones. Ampliar más (solo canal, 42K) degrada el desempeño; exigir más (universos de las Secciones 15/17) lo encarece sin mejorarlo.

**Recomendación operativa** (umbrales OOF, modelo de 24 variables): con el 10% de leads mejor rankeados se captura ~41% de conversiones (lift 4.1×); recall 0.70 cuesta ~29–30% de los leads con precisión ~7.2%; recall 0.60 cuesta ~21% con precisión ~8.9% (2.9× base); recall 0.50 cuesta ~14.5% con precisión ~10.6% (3.5× base). El pipeline serializado (`modelo_final_busqueda.pkl`) incluye features, requisitos de universo y umbrales.

> **Nota de honestidad metodológica.** La selección de configuraciones usó CV sobre el mismo dataset (riesgo de sobre-selección); como mitigación, todos los números reportados se **confirmaron con una segunda semilla de CV** y el patrón (canal ≫ resto) es consistente entre metodologías independientes (univariado, forward, backward, ablación, permutación, SHAP). La validación definitiva sería con leads futuros (out-of-time), que no existen aún en este corte de datos.

---

## Sección 22 — Matriz de confusión y recall total del modelo final

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
Hasta aquí el modelo entrega una <b>probabilidad</b> por lead. Para usarlo en la práctica hay que convertir esa probabilidad en una decisión sí/no, aplicando el umbral operativo (recall ≥ 0.70) ya definido en la §21.4, y viendo la matriz de confusión y el recall sobre <b>todo</b> el universo H.
</div>

In [ ]:
# ── §22  Matriz de confusión + recall TOTAL del modelo final ─────────────
# Umbral operativo: el menor umbral que logra recall >= 0.70 sobre las
# probabilidades OOF del universo H (mismo criterio de la §21.4).
prc_f, rcc_f, thr_f = precision_recall_curve(y_H, prob_H)
i70 = np.where(rcc_f[:-1] >= 0.70)[0][-1]
umbral_op = thr_f[i70]

# Decisión sí/no por lead y conteo de aciertos/errores en TODO el universo
y_pred_H = (prob_H >= umbral_op).astype(int)
cm = confusion_matrix(y_H, y_pred_H)
vn, fp, fn, vp = cm.ravel()

recall_total  = vp / (vp + fn)          # sobre todo el universo, no @K
precision_op  = vp / (vp + fp)
especificidad = vn / (vn + fp)
exactitud     = (vp + vn) / cm.sum()
f1_op         = 2 * precision_op * recall_total / (precision_op + recall_total)

print(f"Umbral operativo: {umbral_op:.4f}")
print(f"Leads marcados 'contactar': {int(y_pred_H.sum()):,} de {len(y_H):,} "
      f"({y_pred_H.mean()*100:.1f}% del universo)")
print(f"VN={vn:,} | FP={fp:,} | FN={fn:,} | VP={vp:,}")

met_cm = pd.DataFrame([
    {'métrica': 'Recall TOTAL  VP/(VP+FN)', 'valor': round(recall_total, 4),
     'lectura': f'detecta {vp} de {vp+fn} conversiones reales del universo'},
    {'métrica': 'Precisión     VP/(VP+FP)', 'valor': round(precision_op, 4),
     'lectura': f'{precision_op*100:.1f}% de los contactados convierte (base: {y_H.mean()*100:.2f}%)'},
    {'métrica': 'Especificidad VN/(VN+FP)', 'valor': round(especificidad, 4),
     'lectura': f'descarta correctamente al {especificidad*100:.1f}% de los que no convierten'},
    {'métrica': 'Exactitud', 'valor': round(exactitud, 4),
     'lectura': 'poco informativa con clases desbalanceadas (3% de positivos)'},
    {'métrica': 'F1', 'valor': round(f1_op, 4),
     'lectura': 'media armónica de precisión y recall'},
])
print("\nMÉTRICAS GLOBALES AL UMBRAL OPERATIVO (todo el universo H):")
print(met_cm.to_string(index=False))
met_cm.to_csv(RESULTS_DIR/'modelo_final_matriz_confusion_metricas.csv', index=False)

# Mapa de calor: color = % dentro de cada fila (si no, el desbalance 97/3
# dejaría casi invisibles las celdas de la clase positiva)
cm_rowpct = cm / cm.sum(axis=1, keepdims=True)
etiquetas = np.array([
    [f"VN\n{vn:,}\n({vn/(vn+fp)*100:.1f}% de los reales 'No')",
     f"FP\n{fp:,}\n({fp/(vn+fp)*100:.1f}% de los reales 'No')"],
    [f"FN\n{fn:,}\n({fn/(fn+vp)*100:.1f}% de los reales 'Sí')",
     f"VP\n{vp:,}\n({vp/(fn+vp)*100:.1f}% de los reales 'Sí')"]])

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(cm_rowpct, annot=etiquetas, fmt='', cmap='Blues', vmin=0, vmax=1,
            cbar=False, linewidths=2, linecolor='white',
            annot_kws={'fontsize': 11}, ax=ax,
            xticklabels=['No convierte', 'Convierte'],
            yticklabels=['No convirtió', 'Convirtió'])
ax.set_xlabel('Predicción del modelo', fontweight='bold')
ax.set_ylabel('Realidad', fontweight='bold')
ax.set_title(f'Matriz de confusión — modelo final (universo H, OOF)\n'
             f'umbral={umbral_op:.4f} | recall total={recall_total:.3f} | '
             f'precisión={precision_op:.3f}', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR/'modelo_final_matriz_confusion.png', dpi=140, bbox_inches='tight')
plt.show()


---

## Sección 23 — Importancia consolidada de variables del modelo final

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
Se consolida en una sola vista qué variables usa el modelo final y en qué dirección actúan, reutilizando la importancia por permutación (§21.6) junto con los coeficientes (odds ratio) del modelo entrenado.
</div>

In [ ]:
# ── §23  Importancia consolidada de variables del modelo final ───────────
# Reutiliza imp_H (permutación) y pipe_H (ya entrenado) de la §21.6.
top_imp = imp_H.head(12).iloc[::-1]

coef_df = pd.DataFrame({
    'feature': [n.replace('cat__', '').replace('num__', '') for n in names_H],
    'coef': pipe_H.named_steps['clf'].coef_[0]})
coef_df['odds_ratio'] = np.exp(coef_df['coef'])
coef_df = coef_df.sort_values('coef', ascending=False).reset_index(drop=True)
top_dir = pd.concat([coef_df.head(8), coef_df.tail(8)])
top_dir = top_dir[~top_dir.index.duplicated()].sort_values('coef')

print("TOP 12 VARIABLES POR IMPORTANCIA (permutación, caída de PR-AUC):")
print(imp_H.head(12).round(4).to_string(index=False))
print("\nCARACTERÍSTICAS CON MAYOR EFECTO (coeficientes del modelo final):")
print(top_dir.iloc[::-1].round(4).to_string(index=False))
coef_df.to_csv(RESULTS_DIR/'modelo_final_coeficientes.csv', index=False)

# Solo importancia por permutacion (la direccion de los coeficientes queda en
# modelo_final_coeficientes.csv y en el cuadro de coeficientes del capitulo).
imp_plot = imp_H.sort_values('imp_media')
col = ['#5B8DB8' if m >= 0 else '#c0c0c0' for m in imp_plot['imp_media']]
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(imp_plot['variable'], imp_plot['imp_media'], xerr=imp_plot['imp_std'],
        color=col, edgecolor='white', error_kw=dict(ecolor='#666', lw=1, capsize=2))
ax.axvline(0, color='#333', lw=.8)
for v, m, s in zip(imp_plot['variable'], imp_plot['imp_media'], imp_plot['imp_std']):
    ax.text(max(m + s, 0) + 0.0016, v, f'{m:+.4f}', va='center', ha='left', fontsize=8)
ax.set_xlabel('Caida media de PR-AUC al permutar la variable  (media +/- desv. estandar)')
ax.set_ylabel('Variable de entrada del modelo')
ax.set_xlim(imp_plot['imp_media'].min() - imp_plot['imp_std'].max() - 0.002,
            imp_plot['imp_media'].max() + imp_plot['imp_std'].max() + 0.012)
ax.grid(alpha=.3, axis='x')
plt.tight_layout()
plt.savefig(FIGURES_DIR/'modelo_final_importancia_consolidada.png', dpi=200, bbox_inches='tight')
plt.show()


---

## Sección 24 — Fórmula matemática del modelo final

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
La regresión logística no es una caja negra: es una fórmula cerrada que se puede escribir en papel. Se renderiza aquí la ecuación completa del modelo final con sus coeficientes entrenados (pipe_H, §21.6).
</div>

In [ ]:
# ── §24  La fórmula matemática del modelo final (renderizada) ─────────────
from IPython.display import display, Math, Markdown

# Coeficientes del modelo entrenado (pipe_H, §21.6)
clf_f = pipe_H.named_steps['clf']
beta0 = clf_f.intercept_[0]
betas = clf_f.coef_[0]
nombres_f = [n.replace('cat__', '').replace('num__', '') for n in names_H]

def _tex(s):
    """Nombre de variable seguro para modo texto de LaTeX/KaTeX."""
    s = s.replace('\\', ' ')
    for a, b in [('_', r'\_'), ('%', r'\%'), ('&', r'\&'), ('#', r'\#'),
                 ('$', r'\$'), ('{', r'\{'), ('}', r'\}'), ('^', ' '), ('~', ' ')]:
        s = s.replace(a, b)
    return s

# 1) Forma general (símbolos)
display(Markdown("**Forma general** — regresión logística (función sigmoide):"))
display(Math(r'\hat{P}(\text{conversión}) = \sigma(z) = \frac{1}{1 + e^{-z}}'
             r'\qquad\text{con}\qquad z = \beta_0 + \sum_{i=1}^{%d}\beta_i\,x_i'
             % len(betas)))

# 2) Fórmula completa con los coeficientes reales, ordenada por |beta|.
#    Se parte en bloques de CHUNK términos: KaTeX limita el nº de expansiones
#    y un único bloque con ~80 términos lo supera ("Too many expansions").
orden = list(np.argsort(-np.abs(betas)))
CHUNK = 12
partes = [orden[i:i+CHUNK] for i in range(0, len(orden), CHUNK)]
display(Markdown(f"**Fórmula completa** con los {len(betas)} coeficientes del modelo final "
                 f"(términos ordenados de mayor a menor peso $|\\beta|$; se muestra en "
                 f"{len(partes)} partes por su longitud):"))
for p_i, part in enumerate(partes):
    filas = [r'z = {}& %+.4f \\' % beta0] if p_i == 0 else []  # intercepto en la 1ª parte
    for i in part:
        filas.append(r'& %+.4f\;\text{%s} \\' % (betas[i], _tex(nombres_f[i])))
    display(Math(r'\begin{aligned}' + '\n' + '\n'.join(filas) + '\n' + r'\end{aligned}'))

# Copia en texto plano
lineas_txt = [f"z = {beta0:+.4f}   (intercepto)"] + \
             [f"    {betas[i]:+.4f} * [{nombres_f[i]}]" for i in orden]
(RESULTS_DIR/'modelo_final_formula.txt').write_text(
    "P(conversion) = 1 / (1 + e^(-z))\n\n" + "\n".join(lineas_txt), encoding='utf-8')

# 3) Ejemplo aplicado a un lead real, también como fórmula
i_lead = int(np.argmax(prob_H))                       # el lead mejor rankeado
x_fila = pipe_H.named_steps['prep'].transform(df_H[FEATS_H].iloc[[i_lead]])[0]
z_lead = beta0 + float(x_fila @ betas)
p_lead = 1 / (1 + np.exp(-z_lead))
p_pipe = pipe_H.predict_proba(df_H[FEATS_H].iloc[[i_lead]])[0, 1]

contrib = x_fila * betas
activos = np.where(np.abs(contrib) > 1e-9)[0]
activos = activos[np.argsort(-np.abs(contrib[activos]))]
filas = [r'z = {}& %+.4f & \text{(intercepto)} \\' % beta0]
for j in activos[:10]:
    filas.append(r'& %+.4f \cdot %.3f & \text{(%s)} \\'
                 % (betas[j], x_fila[j], _tex(nombres_f[j])))
if len(activos) > 10:
    filas.append(r'& %+.4f & \text{(suma de los %d términos restantes)} \\'
                 % (contrib[activos[10:]].sum(), len(activos) - 10))
filas.append(r'= {}& %.4f & \\' % z_lead)
display(Markdown(f"**Ejemplo** — el lead mejor rankeado del universo H (índice {i_lead}). "
                 f"Solo se activan {len(activos)} de {len(betas)} términos (el resto vale 0):"))
display(Math(r'\begin{aligned}' + '\n' + '\n'.join(filas) + '\n' + r'\end{aligned}'))
display(Math(r'\hat{P}(\text{conversión}) = \frac{1}{1 + e^{-(%.4f)}} = %.4f'
             % (z_lead, p_lead)))

# Verificación: la fórmula reproduce exactamente el pipeline
print(f"Verificación:  fórmula = {p_lead:.6f}   |   pipeline.predict_proba = {p_pipe:.6f}"
      f"   |   diferencia = {abs(p_lead - p_pipe):.1e}")
assert abs(p_lead - p_pipe) < 1e-9, "La fórmula debe coincidir con el pipeline"
print("La fórmula reproduce exactamente el modelo.")


## Comparación con una red neuronal (nota breve)

En el notebook de trabajo original se entrenó también una red neuronal (perceptrón multicapa) sobre el mismo Universo H, como punto de comparación. Su desempeño (PR-AUC) resultó equivalente o inferior al de la regresión logística, sin mejorar sobre las interacciones ya construidas manualmente (`edad×ingreso`, `edad×inversión`, etc.), y con las desventajas adicionales de menor interpretabilidad (no existe una fórmula cerrada como la de la sección anterior) y peor calibración de probabilidades. Por esas razones, **el modelo final del proyecto es la Regresión Logística** presentada arriba; la comparación completa con la red neuronal se omite aquí por no ser el modelo seleccionado.


---

## Sección 26 — Métricas consolidadas del modelo final (OOF)

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
A lo largo de las Secciones 21 a 24 el modelo final se evaluó con distintas métricas, pero repartidas en varias celdas. Aquí se reúnen todas en una sola tabla — discriminación (PR-AUC, ROC-AUC), calibración (Brier score) y desempeño al umbral operativo (accuracy, F1, F-beta, MCC, kappa) — sobre las mismas probabilidades OOF del universo H usadas en toda la Parte III.
</div>

In [ ]:
# ── §26  Métricas consolidadas del modelo final (universo H, OOF) ─────────
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
    accuracy_score, f1_score, fbeta_score, balanced_accuracy_score,
    matthews_corrcoef, cohen_kappa_score, precision_recall_curve, confusion_matrix)

# Umbral operativo: menor umbral con recall >= 0.70 sobre las probs OOF (mismo criterio §21.4/§22)
prc_c, rcc_c, thr_c = precision_recall_curve(y_H, prob_H)
i70_c = np.where(rcc_c[:-1] >= 0.70)[0][-1]
umbral_c = thr_c[i70_c]
y_pred_c = (prob_H >= umbral_c).astype(int)

metricas_consolidadas = pd.DataFrame([
    {'métrica': 'PR-AUC',            'valor': round(average_precision_score(y_H, prob_H), 4)},
    {'métrica': 'ROC-AUC',           'valor': round(roc_auc_score(y_H, prob_H), 4)},
    {'métrica': 'Brier score',       'valor': round(brier_score_loss(y_H, prob_H), 4)},
    {'métrica': 'Accuracy',          'valor': round(accuracy_score(y_H, y_pred_c), 4)},
    {'métrica': 'Balanced accuracy', 'valor': round(balanced_accuracy_score(y_H, y_pred_c), 4)},
    {'métrica': 'F1',                'valor': round(f1_score(y_H, y_pred_c), 4)},
    {'métrica': 'F2 (recall-weighted)', 'valor': round(fbeta_score(y_H, y_pred_c, beta=2), 4)},
    {'métrica': 'MCC',               'valor': round(matthews_corrcoef(y_H, y_pred_c), 4)},
    {'métrica': "Cohen's kappa",     'valor': round(cohen_kappa_score(y_H, y_pred_c), 4)},
])
print(f"Umbral operativo (recall >= 0.70): {umbral_c:.4f}\n")
print("MÉTRICAS CONSOLIDADAS — MODELO FINAL (universo H, OOF):")
print(metricas_consolidadas.to_string(index=False))
metricas_consolidadas.to_csv(RESULTS_DIR/'modelo_final_metricas_consolidadas.csv', index=False)

---

## Sección 27 — Modelo final vs. selección aleatoria (Monte Carlo)

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:6px; color:#212121; margin:6px 0;">
<b style="color:#1565C0;">Qué se hizo (explicación simple)</b><br><br>
Se replica la simulación Monte Carlo de la Sección 9.3, pero ahora sobre el modelo final (universo H). La pregunta de negocio es: si ventas solo puede atender un Top X% de los leads, ¿cuántas conversiones reales se capturan usando el modelo, frente a elegir ese mismo % de leads al azar? Se comparan contra un intervalo de confianza del 95% de la selección aleatoria (1000 iteraciones).
</div>

In [ ]:
# ── §27  Modelo final vs selección aleatoria (Monte Carlo) ────────────────
N_MC = 1000; rng = np.random.default_rng(SEED)
p = prob_H; yt = y_H
order = np.argsort(p)[::-1]; cum = np.cumsum(yt[order]); n = len(yt); base = yt.mean()

mc_rows = []
for k in (0.05, 0.10, 0.20, 0.30):
    tn = int(np.ceil(n * k)); tp_m = int(cum[tn - 1])
    rand = np.array([yt[rng.choice(n, tn, replace=False)].sum() for _ in range(N_MC)])
    mc_rows.append({
        'top_%': int(k * 100), 'n_leads': tn,
        'modelo_conversiones': tp_m, 'modelo_recall': round(tp_m / yt.sum(), 3),
        'mc_media': round(rand.mean(), 2), 'mc_ci95_lo': round(np.percentile(rand, 2.5), 2),
        'mc_ci95_hi': round(np.percentile(rand, 97.5), 2),
        'supera_azar': '***' if tp_m > np.percentile(rand, 97.5) else ''})
mc_final = pd.DataFrame(mc_rows)
print("MODELO FINAL vs. SELECCIÓN ALEATORIA (Monte Carlo, 1000 iteraciones):")
print(mc_final.to_string(index=False))
print("\nNota: *** = el modelo captura más conversiones que el límite superior del IC95%% de elegir al azar.")
mc_final.to_csv(RESULTS_DIR/'modelo_final_montecarlo.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(mc_final['top_%'], mc_final['modelo_conversiones'], 'o-', color='#1565C0',
        lw=2, label='Modelo final')
ax.plot(mc_final['top_%'], mc_final['mc_media'], 's--', color='#9E9E9E',
        lw=2, label='Selección aleatoria (media)')
ax.fill_between(mc_final['top_%'], mc_final['mc_ci95_lo'], mc_final['mc_ci95_hi'],
                 color='#9E9E9E', alpha=.25, label='IC 95% aleatorio')
ax.set_xlabel('Top % de leads contactados', fontweight='bold')
ax.set_ylabel('Conversiones capturadas', fontweight='bold')
ax.set_title('Modelo final vs. selección aleatoria (universo H, OOF)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR/'modelo_final_montecarlo.png', dpi=140, bbox_inches='tight')
plt.show()